In [0]:
# CELL 1: Parameters and Read
dbutils.widgets.text("week_ending_date", "24.09.2026")
report_date = dbutils.widgets.get("week_ending_date")

from pyspark.sql.functions import col, lit, current_timestamp, to_date, expr
from datetime import datetime
start_time = datetime.now()

# Read only the specific week for incremental processing
df_report_raw = spark.read.table("default.bronze_pbs_report").filter(col("week_ending_date") == report_date)
df_annex_raw = spark.read.table("default.bronze_pbs_annex").filter(col("week_ending_date") == report_date)

In [0]:
# CELL 2: Clean and Cast Quintile Data
# try_cast converts text like "The weekly SPI with base..." into NULLs, which we then filter out.
r_c1, r_c2, r_c3 = df_report_raw.columns[1], df_report_raw.columns[2], df_report_raw.columns[3]

df_quintile = df_report_raw.select(
    col(df_report_raw.columns[0]).alias("quintile"),
    expr(f"try_cast(`{r_c1}` as double)").alias("index_value"),
    expr(f"try_cast(`{r_c2}` as double)").alias("pct_change_wow"),
    expr(f"try_cast(`{r_c3}` as double)").alias("pct_change_yoy"),
    to_date(col("week_ending_date"), "dd.MM.yyyy").alias("week_ending_date")
).filter(col("quintile").isNotNull() & col("index_value").isNotNull()) \
 .withColumn("load_timestamp", current_timestamp())

print("--- Cleaned Quintile Data ---")
df_quintile.printSchema()
display(df_quintile.limit(5))

--- Cleaned Quintile Data ---
root
 |-- quintile: string (nullable = true)
 |-- index_value: double (nullable = true)
 |-- pct_change_wow: double (nullable = true)
 |-- pct_change_yoy: double (nullable = true)
 |-- week_ending_date: date (nullable = true)
 |-- load_timestamp: timestamp (nullable = false)



quintile,index_value,pct_change_wow,pct_change_yoy,week_ending_date,load_timestamp
nan,NaN,NaN,NaN,2026-09-24,2026-10-07T18:53:52.171Z
Government of Pakistan,NaN,NaN,NaN,2026-09-24,2026-10-07T18:53:52.171Z
nan,NaN,NaN,NaN,2026-09-24,2026-10-07T18:53:52.171Z
nan,NaN,NaN,NaN,2026-09-24,2026-10-07T18:53:52.171Z
nan,NaN,NaN,NaN,2026-09-24,2026-10-07T18:53:52.171Z


In [0]:
# CELL 3: Clean and Cast City Prices
a_c2, a_c3, a_c4 = df_annex_raw.columns[2], df_annex_raw.columns[3], df_annex_raw.columns[4]

df_prices = df_annex_raw.select(
    col(df_annex_raw.columns[0]).alias("item_name"),
    col(df_annex_raw.columns[1]).alias("city"),
    expr(f"try_cast(`{a_c2}` as double)").alias("min_price"),
    expr(f"try_cast(`{a_c3}` as double)").alias("max_price"),
    expr(f"try_cast(`{a_c4}` as double)").alias("avg_price"),
    to_date(col("week_ending_date"), "dd.MM.yyyy").alias("week_ending_date")
).filter(col("item_name").isNotNull() & col("city").isNotNull() & col("avg_price").isNotNull())

df_prices = df_prices.withColumn("price_dispersion_pct", (col("max_price") - col("min_price")) / col("avg_price")) \
                     .withColumn("load_timestamp", current_timestamp())

print("--- Cleaned City Prices ---")
display(df_prices.limit(5))

--- Cleaned City Prices ---


item_name,city,min_price,max_price,avg_price,week_ending_date,price_dispersion_pct,load_timestamp
nan,nan,NaN,null,NaN,2026-09-24,null,2026-10-07T18:54:14.910Z
SL.,nan,NaN,null,NaN,2026-09-24,null,2026-10-07T18:54:14.910Z
1,2,3.0,4.0,5.0,2026-09-24,0.2,2026-10-07T18:54:14.910Z
nan,nan,NaN,null,NaN,2026-09-24,null,2026-10-07T18:54:14.910Z
1,Wheat Flour Bag,null,2933.33,2993.14,2026-09-24,null,2026-10-07T18:54:14.910Z


In [0]:
# CELL 4: Write to Silver
import uuid as _uuid

formatted_date = datetime.strptime(report_date, "%d.%m.%Y").strftime("%Y-%m-%d")

try:
    df_quintile.write.format("delta").mode("overwrite").option("replaceWhere", f"week_ending_date = '{formatted_date}'").saveAsTable("default.silver_quintile_index")
    df_prices.write.format("delta").mode("overwrite").option("replaceWhere", f"week_ending_date = '{formatted_date}'").saveAsTable("default.silver_city_price_observation")

    rows = df_quintile.count() + df_prices.count()
    spark.sql(f"INSERT INTO default.pipeline_execution_logs VALUES ('{_uuid.uuid4()}', 'Bronze-to-Silver-PBS', 'Week_{report_date}', '{start_time}', current_timestamp(), 'Success', {rows}, '')")
except Exception as e:
    spark.sql(f"INSERT INTO default.pipeline_execution_logs VALUES ('{_uuid.uuid4()}', 'Bronze-to-Silver-PBS', 'Week_{report_date}', '{start_time}', current_timestamp(), 'Failure', 0, '{str(e)[0:200]}')")
    raise e